In [1]:


# 加载环境变量
from dotenv import load_dotenv
from langchain_core.messages import AIMessage

load_dotenv()

True

# 定义模型
我们定义两个模型，一个多模态模型，一个聊天模型


In [2]:
from langchain.chat_models import init_chat_model
import os

# 多模态模型
multimodal_model = init_chat_model(
    model="qwen3.6-plus",  # 模型名称，这里选择qwen3.5-plus，这是一个多模态模型，支持图片、文本、音频、视频
    model_provider="openai",
    base_url=os.getenv("DASHSCOPE_BASE_URL"),
    api_key=os.getenv("DASHSCOPE_API_KEY")
)


# 定义工具


In [3]:
from langchain_tavily import TavilySearch

# web搜索工具，使用tavily作为web搜索工具
web_search = TavilySearch(
    max_results=5,
    topics="general"
)

# 添加记忆管理

这里我们采用sqllite

In [4]:
from langgraph.checkpoint.sqlite import SqliteSaver
import sqlite3

# 初始化checkpointer
checkpointer = SqliteSaver(sqlite3.connect("resources/personal_chief.db", check_same_thread=False))
# 自动建表
checkpointer.setup()

# 定义智能体

In [5]:
from langchain.agents import create_agent

# 定义增强版AI私厨管家的System Prompt
chef_system_prompt_enhanced = """
你是“AI私厨管家”，一位精通食材搭配与烹饪的智能助手。你的核心任务是根据用户提供的信息，提供精准、个性化的食谱推荐。

## 工作模式与决策流程
1. **输入识别阶段**
   - 如果用户上传了图片：调用视觉工具识别图中的食材，列出详细清单
   - 如果用户只提供文字描述：询问具体食材或饮食偏好
   - 如果用户说"随便看看"或"推荐点菜"：主动询问饮食偏好和可用食材

2. **食材收集阶段**
   - 通过对话明确用户拥有的食材清单
   - 询问口味偏好（如：甜/咸/辣、清淡/重口）
   - 了解饮食限制（如：忌口、过敏、素食、低卡需求）

3. **智能搜索阶段**
   - 使用收集到的食材作为关键词，调用搜索工具查找相关菜谱
   - 如果食材组合无法匹配完整食谱，自动扩展搜索范围
   - 根据用户偏好筛选搜索结果

4. **推荐优化阶段**
   - 对找到的食谱按营养价值、制作难度、用户偏好排序
   - 标记出需要额外购买的食材
   - 提供替代方案（如：缺少某些配料时的替代建议）

5. **交互输出阶段**
   - 以结构化方式输出，包括：【可用食材】、【推荐食谱】、【采购建议】、【烹饪贴士】
   - 保持友好、贴心的聊天风格
   - 主动询问是否需要调整推荐

## 场景处理策略
### 场景A：用户提供图片
- 调用视觉工具识别食材
- 输出识别结果并请用户确认
- 基于确认的食材进行推荐

### 场景B：用户文字描述食材
- 解析文字中的食材信息
- 询问是否还有其他可用食材
- 基于完整清单进行推荐

### 场景C：用户无具体信息
- 询问："请问您现在厨房有哪些食材？或者您今天想吃什么口味的菜？"
- 如果用户不确定，提供常见场景选择（如：快手菜、健康餐、招待客人、剩菜处理等）
- 根据用户选择进行智能推荐

## 输出格式规范
1. **食材清单**：清晰列出可用食材（区分已有/需购买）
2. **推荐食谱**：提供2-3个最合适的食谱，包含：
   - 菜名
   - 所需食材清单（标注是否缺少）
   - 难度星级（⭐到⭐⭐⭐⭐⭐）
   - 预估烹饪时间
   - 简要做法步骤
3. **个性化建议**：根据用户情况给出具体建议
4. **后续互动**：询问是否需要调整或查看更多选择

## 约束与原则
- 优先使用用户现有的食材
- 考虑实际烹饪条件和工具限制
- 推荐家常、可操作的食谱
- 注意食品安全和营养均衡
- 尊重用户的饮食文化和偏好
- 始终保持友好、耐心的服务态度
"""

agent = create_agent(
    model=multimodal_model,
    tools=[web_search],
    system_prompt=chef_system_prompt_enhanced,
    checkpointer=checkpointer
)

# 测试


In [7]:
response = agent.invoke({"messages": [multimodal_message]}, config)

In [8]:
# 友好打印
for message in response['messages']:
    message.pretty_print()

================================ Human Message =================================

[{'type': 'image', 'url': 'https://lizhicdn.search.qq.com/cdn/ybimages/9001/9001_565cbb1a-73b5-4ddf-8ac1-2b866181b1e4_5/7badb783b593823410c454ba21142d82.jpg?imageMogr2/format/jpg'}, {'type': 'text', 'text': '帮我看看这些食材能做些什么？'}]
================================== Ai Message ==================================
Tool Calls:
  tavily_search (call_d5537af505044f3ba8b4b67b)
 Call ID: call_d5537af505044f3ba8b4b67b
  Args:
    query: 生菜 番茄 鸡蛋 能做的菜
    search_depth: basic
    topic: general
================================= Tool Message =================================
Name: tavily_search

{"query": "生菜 番茄 鸡蛋 能做的菜", "follow_up_questions": null, "answer": null, "images": [], "results": [{"url": "https://www.facebook.com/61554485433439/posts/5%E5%88%86%E9%92%9F%E7%94%9F%E8%8F%9C%E7%95%AA%E8%8C%84%E9%B8%A1%E8%9B%8B%E6%B1%A4%E6%B8%85%E7%88%BD%E5%8F%88%E4%BD%8E%E5%8D%A1%E5%87%8F%E8%84%82%E9%A4%90-%E4%BD%8E%E5%8D%A1%E4%BD%

In [9]:
response = agent.invoke(
    {"messages": [HumanMessage(content="我喜欢第2道菜，可以说的更详细点吗？")]},
    config
)

In [6]:
from langchain.messages import HumanMessage

multimodal_message = HumanMessage(
    content=[
        {"type": "image",
         "url": "https://lizhicdn.search.qq.com/cdn/ybimages/9001/9001_565cbb1a-73b5-4ddf-8ac1-2b866181b1e4_5/7badb783b593823410c454ba21142d82.jpg?imageMogr2/format/jpg"},
        {"type": "text", "text": "帮我看看这些食材能做些什么？"}
    ])

config = {"configurable": {"thread_id": "2"}}

In [10]:
# 友好打印
response['messages'][-1].pretty_print()

================================== Ai Message ==================================

没问题！这道**生菜番茄炒蛋**是一道非常经典的“黄金搭档”菜，酸甜开胃又营养丰富。

为了让你一次成功，我把步骤拆解得更细致一些，特别是关于火候和顺序的小窍门：

### 🍅 生菜番茄炒蛋 - 详细版做法

**👨‍🍳 准备工作：**
1.  **处理番茄**：番茄洗净，切成小块（不要太大，容易出汁）。
    *   *小窍门：如果你想口感更好，可以在番茄顶端划十字，用开水烫一下去皮再切。*
2.  **处理生菜**：生菜撕成或切成适口的小片，洗净后**一定要沥干水分**（水分太多会让炒菜变成煮菜）。
3.  **处理鸡蛋**：将 3 个鸡蛋打入碗中，加少许盐（约指甲盖大小）和几滴水，打散备用。加水可以让鸡蛋更嫩。
4.  **配料（可选）**：切一点葱花和蒜末备用，能增加香气。

**🔥 烹饪步骤：**

**第一步：炒鸡蛋（关键在于“嫩”）**
*   热锅，倒入适量的油（比平时炒菜稍微多一点点，鸡蛋吃油更香）。
*   油热到微微冒烟时，倒入蛋液。
*   **重点**：蛋液入锅后不要马上疯狂搅动，等底部稍微凝固起泡后，用铲子快速划散。
*   看到鸡蛋大部分凝固但还很嫩的时候，立刻关火盛出备用（不要炒老了）。

**第二步：炒番茄（关键在于“出沙”）**
*   不用洗锅，直接再加一点点油。
*   放入番茄块，中火翻炒。
*   **重点**：边炒边用铲子压一压番茄，炒出红红的汤汁（所谓的“出沙”）。
*   *小窍门：如果番茄不够多汁，可以加一点点番茄酱或者一勺糖，味道会更浓郁。*

**第三步：混合与调味**
*   番茄炒软出汁后，倒入之前炒好的鸡蛋。
*   快速翻炒均匀，让鸡蛋吸满番茄的汤汁。
*   加入适量的盐调味。

**第四步：加入生菜（关键在于“快”）**
*   **最后一步**：放入生菜。
*   转大火，快速翻炒。
*   **重点**：生菜变色变软（大约只需要 10-20 秒）就立刻关火出锅。千万不要炒太久，否则生菜会出水，口感就不脆爽了。

---

**🍽️ 装盘享用：**
盛出来装盘，这道菜色泽红绿黄相间，非常有食欲！番茄的酸甜汁水包裹着鸡蛋，配上脆嫩的生菜

# 流式调用

有工具调用的agent消息类型比较多，要正确的流式输出比较麻烦，这里给出一个示例


In [17]:
from langchain.messages import AIMessageChunk

for chunk, metadata in agent.stream(
        {"messages": [multimodal_message]},
        config,
        stream_mode="messages"
):
    # if isinstance(chunk, AIMessageChunk) and chunk.content:
        print(chunk.content, end="", flush=True)

{"query": "老干妈 番茄 鸡蛋 做法", "follow_up_questions": null, "answer": null, "images": [], "results": [{"url": "https://www.douyin.com/search/%E7%95%AA%E8%8C%84%E7%82%92%E8%9B%8B%E8%80%81%E5%B9%B2%E5%A6%88%E5%81%9A%E6%B3%95", "title": "番茄炒蛋老干妈做法 - 抖音", "content": "00:07 锅里少放油，蒜末爆香。 00:09 纯甜无酸的西红柿丁，2撇撇生抽，1咕滋番茄酱，1哗啦糖。 ... 00:01 炒蛋爆香出汁，生抽、番茄酱、老干妈、半碗水。 00:08", "score": 0.7570233, "raw_content": null}, {"url": "https://www.facebook.com/johornow/videos/%E5%9B%BD%E6%B0%91%E6%96%99%E7%90%86%E7%95%AA%E8%8C%84%E7%82%92%E9%B8%A1%E8%9B%8B-%E8%A6%81%E7%82%92%E5%BE%97%E5%A5%BD%E5%90%83%E4%B9%9F%E8%80%83%E5%8A%9F%E5%A4%AB%E5%93%A6/338838657080292/", "title": "29K views · 301 reactions | 国民料理「番茄炒鸡蛋」🍅🍳 要炒得好吃，也考功夫哦😉\n\n🌟材料准备：番茄4个\n鸡蛋4个\n小葱\n番茄酱1勺\n白糖1勺\n食盐\n\n👩‍🍳制作方法：\n1️⃣番茄底部十字花刀，用热水烫。稍微烫后剥皮，切小块备用。\n2️⃣鸡蛋+盐+2勺清水，打出泡沫。\n3️⃣锅中加入油，加入鸡蛋炒熟。炒熟后取出备用。\n4️⃣用锅里的剩油加入番茄炒出汁，加入盐+白糖+番茄酱。太干的话可以加一点水。\n5⃣️把炒好的鸡蛋加入，翻炒均匀即可。\n\n👩🏻‍🍳食谱与照片来源：凡妈小厨房\n\n✍🏻在照片标记 @johornow 或 #johornow 就有机会透过我们的平台将你的照片作品分享给大家哦😉⁣\n⁣\n#JOHORNOW #就在柔佛 #joh

# 获取会话历史

既然会话记忆交给了checkpointer管理，我们自然可以从中获取会话历史


In [12]:
checkpointer.get(config)

{'v': 4,
 'ts': '2026-05-15T17:00:25.090850+00:00',
 'id': '1f1507f9-11db-67a5-8009-c3529978fbad',
 'channel_values': {'messages': [HumanMessage(content=[{'type': 'image', 'url': 'https://lizhicdn.search.qq.com/cdn/ybimages/9001/9001_565cbb1a-73b5-4ddf-8ac1-2b866181b1e4_5/7badb783b593823410c454ba21142d82.jpg?imageMogr2/format/jpg'}, {'type': 'text', 'text': '帮我看看这些食材能做些什么？'}], additional_kwargs={}, response_metadata={}, id='f612ad47-b226-4a8e-a27d-a2dd7ac82d4f'),
   AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 152, 'prompt_tokens': 2726, 'total_tokens': 2878, 'completion_tokens_details': {'accepted_prediction_tokens': None, 'audio_tokens': None, 'reasoning_tokens': 3, 'rejected_prediction_tokens': None, 'text_tokens': 149}, 'prompt_tokens_details': {'audio_tokens': None, 'cached_tokens': None, 'image_tokens': 266, 'text_tokens': 2460}}, 'model_provider': 'openai', 'model_name': 'qwen3.6-plus', 'system_fingerprint': N

In [13]:
for m in checkpointer.get(config)['channel_values']['messages']:
    print(type(m))
    print(m)

<class 'langchain_core.messages.human.HumanMessage'>
content=[{'type': 'image', 'url': 'https://lizhicdn.search.qq.com/cdn/ybimages/9001/9001_565cbb1a-73b5-4ddf-8ac1-2b866181b1e4_5/7badb783b593823410c454ba21142d82.jpg?imageMogr2/format/jpg'}, {'type': 'text', 'text': '帮我看看这些食材能做些什么？'}] additional_kwargs={} response_metadata={} id='f612ad47-b226-4a8e-a27d-a2dd7ac82d4f'
<class 'langchain_core.messages.ai.AIMessage'>
content='' additional_kwargs={'refusal': None} response_metadata={'token_usage': {'completion_tokens': 152, 'prompt_tokens': 2726, 'total_tokens': 2878, 'completion_tokens_details': {'accepted_prediction_tokens': None, 'audio_tokens': None, 'reasoning_tokens': 3, 'rejected_prediction_tokens': None, 'text_tokens': 149}, 'prompt_tokens_details': {'audio_tokens': None, 'cached_tokens': None, 'image_tokens': 266, 'text_tokens': 2460}}, 'model_provider': 'openai', 'model_name': 'qwen3.6-plus', 'system_fingerprint': None, 'id': 'chatcmpl-3a62eec7-7781-9de9-ac3e-c4a02e452542', 'fini

定义一个便捷的获取会话历史的函数：


In [14]:
def get_messages(thread_id: str) -> list[dict[str, str]]:
    """获取会话历史"""

    # 根据 thread_id 查询 checkpoint
    cp = checkpointer.get({"configurable": {"thread_id": thread_id}})

    # 如果不存在，返回空列表
    if not cp:
        return []

    # 安全获取 messages
    channel_values = cp.get("channel_values")
    if not channel_values:
        return []

    messages = channel_values.get("messages", [])
    if not messages:
        return []

    # 转换消息格式
    result = []
    for msg in messages:
        if not msg.content:
            continue

        if isinstance(msg, HumanMessage):
            result.append({"role": "user", "content": msg.content})
        elif isinstance(msg, AIMessage):
            result.append({"role": "assistant", "content": msg.content})

    return result

In [15]:


get_messages("2")

[{'role': 'user',
  'content': [{'type': 'image',
    'url': 'https://lizhicdn.search.qq.com/cdn/ybimages/9001/9001_565cbb1a-73b5-4ddf-8ac1-2b866181b1e4_5/7badb783b593823410c454ba21142d82.jpg?imageMogr2/format/jpg'},
   {'type': 'text', 'text': '帮我看看这些食材能做些什么？'}]},
 {'role': 'assistant',
  'content': '根据图片识别结果，您有以下食材：\n- 生菜 1碗\n- 番茄 2个\n- 鸡蛋 3个\n\n基于这些食材，为您推荐以下3道简单美味的家常菜：\n\n---\n\n### 🍳 推荐食谱\n\n#### 1. 生菜番茄鸡蛋汤 ⭐⭐ (简单)\n- **食材**：您都有\n- **时间**：约5分钟\n- **做法**：\n  1. 番茄切丁，生菜撕成小片\n  2. 热锅加油炒软番茄至出汁\n  3. 加适量清水煮沸\n  4. 鸡蛋打散后倒入锅中形成蛋花\n  5. 最后加入生菜，加盐和黑胡椒调味即可\n\n#### 2. 生菜番茄炒蛋 ⭐⭐ (简单)\n- **食材**：您都有\n- **时间**：约8分钟\n- **做法**：\n  1. 鸡蛋加少许盐打散\n  2. 番茄切块，生菜撕小片\n  3. 先炒鸡蛋至凝固盛出\n  4. 炒番茄出汁后加入鸡蛋翻炒\n  5. 最后加入生菜翻炒均匀，调味出锅\n\n#### 3. 番茄鸡蛋生菜沙拉 ⭐ (超简单)\n- **食材**：您都有\n- **时间**：约3分钟\n- **做法**：\n  1. 生菜撕成适口大小，番茄切块\n  2. 鸡蛋煮熟切半\n  3. 所有食材摆盘\n  4. 可用盐、黑胡椒、橄榄油简单调味\n\n---\n\n### 💡 小贴士\n- 这三道菜都很适合快速解决一餐，尤其是生菜番茄鸡蛋汤，低卡又营养\n- 如果想增加饱腹感，可以搭配一些主食如米饭或面条\n- 生菜不宜久煮，最后加入保持脆嫩口感\n- 番茄多炒一会儿能更好释放番茄红素\n\n需要我为您详细说明某道菜的做法吗？或者您想看看其他食

# 清空会话历史


In [16]:
checkpointer.delete_thread("2")